In [ ]:
import glob
import os

import numpy as np
import rasterio
from matplotlib import pyplot as plt

# Visualize samples directly from the folder structure (pre-LMDB)
#
# Note:
# - Cloud/gap pixels are encoded as NaN in the GeoTIFFs.
# - For visualization, people often divide by 2500, but that can look grey-ish.
#   Here we use a robust percentile stretch to get more natural contrast.

_s2_data_dir = os.environ.get(
    "S2_DATA_DIR",
    "../../../sentinel/data",
)
output_dir = os.path.join(_s2_data_dir, "SSL-consecutive")

n_samples = 10
n_timesteps = 5
seed = None  # set to None for non-deterministic sampling
TARGET_T = n_timesteps // 2


def robust_rgb(b4: np.ndarray, b3: np.ndarray, b2: np.ndarray,
               pmin: float = 2, pmax: float = 98,
               lo: float = None, hi: float = None):
    rgb = np.stack([b4, b3, b2], axis=-1).astype(np.float32)

    # Gap mask: any channel is NaN
    gaps = np.any(np.isnan(rgb), axis=-1)

    # Robust stretch (NaN pixels are already excluded by nanpercentile)
    if lo is None or hi is None:
        lo = np.nanpercentile(rgb, pmin)
        hi = np.nanpercentile(rgb, pmax)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = 0.0, 2500.0

    rgb = (rgb - lo) / (hi - lo + 1e-6)
    rgb = np.clip(rgb, 0, 1)
    rgb[gaps] = 0.0
    return rgb, gaps


# Randomly choose samples without collecting all subdirectories (reservoir sampling)
rng = np.random.default_rng(seed) if seed is not None else np.random.default_rng()

chosen_sample_dirs = []
seen = 0
with os.scandir(output_dir) as it:
    for entry in it:
        if not entry.is_dir():
            continue
        seen += 1
        if len(chosen_sample_dirs) < n_samples:
            chosen_sample_dirs.append(entry.path)
        else:
            j = int(rng.integers(0, seen))
            if j < n_samples:
                chosen_sample_dirs[j] = entry.path

if len(chosen_sample_dirs) == 0:
    raise FileNotFoundError(f"No sample directories found under: {output_dir}")

# Optional: keep a stable order in the plot grid (does not affect sampling)
chosen_sample_dirs = sorted(chosen_sample_dirs)

fig, axes = plt.subplots(
    len(chosen_sample_dirs), n_timesteps, figsize=(20, 4 * len(chosen_sample_dirs))
)
if len(chosen_sample_dirs) == 1:
    axes = [axes]  # Ensure axes is always 2D

for i, sample_dir in enumerate(chosen_sample_dirs):
    # Each timestep folder is named by the image "system:index"
    timestep_dirs = sorted(
        d for d in glob.glob(os.path.join(sample_dir, "*")) if os.path.isdir(d)
    )

    # Pass 1: read raw bands for all timesteps and compute shared normalization range
    all_raw = []
    raw_bands = {}
    for t, tdir in enumerate(timestep_dirs[:n_timesteps]):
        try:
            with rasterio.open(os.path.join(tdir, "S2C.tif")) as src:
                b2 = src.read(1).astype(np.float32)
                b3 = src.read(2).astype(np.float32)
                b4 = src.read(3).astype(np.float32)
            raw_bands[t] = (b4, b3, b2)
            for arr in (b4, b3, b2):
                valid = arr[~np.isnan(arr)]
                if valid.size:
                    all_raw.append(valid)
        except Exception:
            raw_bands[t] = None

    combined = np.concatenate(all_raw) if all_raw else np.array([0.0, 1.0])
    row_lo = float(np.percentile(combined, 2))
    row_hi = float(np.percentile(combined, 98))
    if not np.isfinite(row_lo) or not np.isfinite(row_hi) or row_hi <= row_lo:
        row_lo, row_hi = 0.0, 2500.0

    # Pass 2: display each timestep using the shared normalization range
    for t in range(n_timesteps):
        ax = axes[i][t] if len(chosen_sample_dirs) > 1 else axes[0][t]

        if t >= len(timestep_dirs) or raw_bands.get(t) is None:
            ax.imshow(np.zeros((10, 10, 3)))
            ax.set_title(f"Sample {i+1}, t={t}\n(missing)")
            ax.axis("off")
            continue

        tdir = timestep_dirs[t]
        timestep_name = os.path.basename(tdir)[:8]  # YYYYMMDD
        timestep_name = f"{timestep_name[:4]}/{timestep_name[4:6]}/{timestep_name[6:8]}"

        b4, b3, b2 = raw_bands[t]
        h, w = b4.shape
        rgb, gaps = robust_rgb(b4, b3, b2, lo=row_lo, hi=row_hi)

        ax.imshow(rgb)

        size_str = f"{h}×{w}"
        frac = 100.0 * float(gaps.sum()) / float(gaps.size)

        title = f"Sample {i+1}, t={t}\n{timestep_name}  {size_str}  gaps={frac:.1f}%"
        if t == TARGET_T:
            title += " (target)"
        ax.set_title(title)
        ax.axis("off")

plt.tight_layout()
plt.show()